In [1]:
from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="This is the main text content I am using to create RAG",\
    metadata={
        "source":"example.txt",
        "pages": 1,
        "author": "Guruganesh",
        "date_created": "2026-13-06"
        }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Guruganesh', 'date_created': '2026-13-06'}, page_content='This is the main text content I am using to create RAG')

### Creating a simple text document

In [3]:
import os
os.makedirs("../data/text_files", exist_ok=True) # This creates a folder called "text_files" inside the Data folder

sample_text = {
    "../data/text_files/python_intro.txt": """This is a sample text for the document. 
    Guruganesh created this document on 2026-13-06.

    I love AI and Python! I am excited to learn more about these technologies.

    I will be an AI enthusiast and Python developer real soon.
    
    Hare Krishna Hare Krishna, Krishna Krishna Hare Hare, 
    Hare Rama Hare Rama, Rama Rama Hare Hare."""
}

for file_path, content in sample_text.items():
    with open(file_path,"w",encoding="utf-8") as f:
        f.write(content)

print("Sample text file created!")

Sample text file created!


### TextLoader

In [4]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
document = loader.load()
print(document)


C:\Users\Guruganesh\AppData\Local\Temp\ipykernel_520772\256376635.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\Guruganesh\OneDrive\Desktop\MyLearnings\Practice\LangChainDS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='This is a sample text for the document. \n    Guruganesh created this document on 2026-13-06.\n\n    I love AI and Python! I am excited to learn more about these technologies.\n\n    I will be an AI enthusiast and Python developer real soon.\n\n    Hare Krishna Hare Krishna, Krishna Krishna Hare Hare, \n    Hare Rama Hare Rama, Rama Rama Hare Hare.')]


### PDF Loader

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    "../data/pdf_files/",
    glob="**/*.pdf", # pattern to match PDF files
    loader_cls=PyMuPDFLoader, # loader class to use for loading PDF files
    show_progress=False
)

pdf_documents = dir_loader.load()
print(pdf_documents)

[Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2024-08-26T13:20:13-04:00', 'source': '..\\data\\pdf_files\\Lecture 1 - Introduction to Artificial Intelligence.pdf', 'file_path': '..\\data\\pdf_files\\Lecture 1 - Introduction to Artificial Intelligence.pdf', 'total_pages': 36, 'format': 'PDF 1.7', 'title': 'PowerPoint Presentation', 'author': 'Racicot, Mary', 'subject': '', 'keywords': '', 'moddate': '2024-08-26T13:20:13-04:00', 'trapped': '', 'modDate': "D:20240826132013-04'00'", 'creationDate': "D:20240826132013-04'00'", 'page': 0}, page_content='Worcester Polytechnic Institute\nCS 534\nIntroduction to Artificial Intelligence\nLecture 1: Introduction to AI\nBy\nBen C.K. Ngan'), Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2024-08-26T13:20:13-04:00', 'source': '..\\data\\pdf_files\\Lect

### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [7]:
### Read all PDF files in a directory

def process_all_pdfs(pdf_directory):
    """ 
    Process all PDF files in a given directory and return a list of documents.
    """
    all_documents = []
    pdf_dir = Path(pdf_directory)
    # Find all PDF files recurisevly
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add Information to Metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
        except Exception as e:
            print(f"Error : {e}")
    
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents


# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 5 PDF files to process.

Processing: Lecture 1 - Introduction to Artificial Intelligence.pdf

Processing: Lecture 2 - Intelligent Agents.pdf

Processing: Lecture 3 - Solving Problems by Searching (1).pdf

Processing: Lecture 4 - Search in Complex Environments (1).pdf

Processing: Lecture 5 - Knowledge Representation.pdf
Total documents loaded: 259


In [8]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2024-08-26T13:20:13-04:00', 'title': 'PowerPoint Presentation', 'author': 'Racicot, Mary', 'moddate': '2024-08-26T13:20:13-04:00', 'source': '..\\data\\pdf_files\\Lecture 1 - Introduction to Artificial Intelligence.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1', 'source_file': 'Lecture 1 - Introduction to Artificial Intelligence.pdf', 'file_type': 'pdf'}, page_content='Worcester Polytechnic Institute\nCS 534\nIntroduction to Artificial Intelligence\nLecture 1: Introduction to AI\nBy\nBen C.K. Ngan'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2024-08-26T13:20:13-04:00', 'title': 'PowerPoint Presentation', 'author': 'Racicot, Mary', 'moddate': '2024-08-26T13:20:13-04:00', 'source': '..\\data\\pdf_files\\Lecture 1 - Introduction to 

## Text splitting get into chunks

In [9]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    #Show example of chunk

    if split_docs:
        print(f"\nExample of a chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [10]:
chunks = split_documents(all_pdf_documents)
chunks

Split 259 documents into 274 chunks.

Example of a chunk:
Content: Worcester Polytechnic Institute
CS 534
Introduction to Artificial Intelligence
Lecture 1: Introduction to AI
By
Ben C.K. Ngan...
Metadata: {'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2024-08-26T13:20:13-04:00', 'title': 'PowerPoint Presentation', 'author': 'Racicot, Mary', 'moddate': '2024-08-26T13:20:13-04:00', 'source': '..\\data\\pdf_files\\Lecture 1 - Introduction to Artificial Intelligence.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1', 'source_file': 'Lecture 1 - Introduction to Artificial Intelligence.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2024-08-26T13:20:13-04:00', 'title': 'PowerPoint Presentation', 'author': 'Racicot, Mary', 'moddate': '2024-08-26T13:20:13-04:00', 'source': '..\\data\\pdf_files\\Lecture 1 - Introduction to Artificial Intelligence.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1', 'source_file': 'Lecture 1 - Introduction to Artificial Intelligence.pdf', 'file_type': 'pdf'}, page_content='Worcester Polytechnic Institute\nCS 534\nIntroduction to Artificial Intelligence\nLecture 1: Introduction to AI\nBy\nBen C.K. Ngan'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2024-08-26T13:20:13-04:00', 'title': 'PowerPoint Presentation', 'author': 'Racicot, Mary', 'moddate': '2024-08-26T13:20:13-04:00', 'source': '..\\data\\pdf_files\\Lecture 1 - Introduction to 

### Semantic Chunking

Instead of splitting by a fixed character count, `SemanticChunker` embeds sentences and splits where the semantic similarity between consecutive sentences drops — i.e. where the topic actually shifts. This suits lecture slides better than a fixed size, since some slides pack a dense definition into a few lines while others spread one idea across many bullet points.

Uses a local `sentence-transformers` embedding model (`all-MiniLM-L6-v2`), so no API key is required.

In [11]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

# Local embedding model - runs on CPU, no API key needed
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


def split_documents_semantic(documents, breakpoint_threshold_type="percentile", breakpoint_threshold_amount=95):
    """Split documents into chunks based on semantic similarity between sentences.

    breakpoint_threshold_type: how the split threshold is computed -
        "percentile" (default), "standard_deviation", "interquartile", or "gradient".
    breakpoint_threshold_amount: sensitivity for that threshold type - lower means more (smaller) chunks.
    """
    semantic_splitter = SemanticChunker(
        embeddings,
        breakpoint_threshold_type=breakpoint_threshold_type,
        breakpoint_threshold_amount=breakpoint_threshold_amount,
    )
    split_docs = semantic_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print(f"\nExample of a chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

C:\Users\Guruganesh\AppData\Local\Temp\ipykernel_520772\4099920753.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11804.93it/s]


In [12]:
semantic_chunks = split_documents_semantic(all_pdf_documents)
semantic_chunks

Split 259 documents into 374 chunks.

Example of a chunk:
Content: Worcester Polytechnic Institute
CS 534
Introduction to Artificial Intelligence
Lecture 1: Introduction to AI
By
Ben C.K. Ngan...
Metadata: {'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2024-08-26T13:20:13-04:00', 'title': 'PowerPoint Presentation', 'author': 'Racicot, Mary', 'moddate': '2024-08-26T13:20:13-04:00', 'source': '..\\data\\pdf_files\\Lecture 1 - Introduction to Artificial Intelligence.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1', 'source_file': 'Lecture 1 - Introduction to Artificial Intelligence.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2024-08-26T13:20:13-04:00', 'title': 'PowerPoint Presentation', 'author': 'Racicot, Mary', 'moddate': '2024-08-26T13:20:13-04:00', 'source': '..\\data\\pdf_files\\Lecture 1 - Introduction to Artificial Intelligence.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1', 'source_file': 'Lecture 1 - Introduction to Artificial Intelligence.pdf', 'file_type': 'pdf'}, page_content='Worcester Polytechnic Institute\nCS 534\nIntroduction to Artificial Intelligence\nLecture 1: Introduction to AI\nBy\nBen C.K. Ngan'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2024-08-26T13:20:13-04:00', 'title': 'PowerPoint Presentation', 'author': 'Racicot, Mary', 'moddate': '2024-08-26T13:20:13-04:00', 'source': '..\\data\\pdf_files\\Lecture 1 - Introduction to 

## Embedding + Vector Store

Index `semantic_chunks` into a persistent Chroma vector store using the same `embeddings` model, then wrap it as a retriever and sanity-check it with a test query before building anything on top of it.

In [13]:
from langchain_chroma import Chroma
import shutil

persist_directory = "../data/vector_store/semantic_chunks"

# Clear any existing collection before re-indexing. Chroma.from_documents()
# appends to a persisted collection rather than replacing it, so re-running
# this cell without clearing first silently piles up duplicate chunks
# (each re-run adds another full copy), which crowds out diverse results
# at retrieval time.
shutil.rmtree(persist_directory, ignore_errors=True)

vectorstore = Chroma.from_documents(
    documents=semantic_chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="semantic_chunks",
)

print(f"Indexed {vectorstore._collection.count()} chunks into Chroma at '{persist_directory}'.")

Indexed 374 chunks into Chroma at '../data/vector_store/semantic_chunks'.


In [14]:
# MMR (Maximal Marginal Relevance) trades a bit of pure similarity for diversity:
# it pulls fetch_k candidates, then greedily picks k of them that are relevant
# but not near-duplicates of each other. This matters for topics whose full
# explanation is spread across a couple of different slides/chunks.
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 6, "fetch_k": 20, "lambda_mult": 0.5},
)

In [15]:
# Sanity-check retrieval before building anything on top of it
test_query = "What is an intelligent agent?"
results = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(results)} chunks:\n")
for i, doc in enumerate(results, start=1):
    print(f"--- Result {i} (source: {doc.metadata.get('source_file')}, page: {doc.metadata.get('page')}) ---")
    print(doc.page_content[:300])
    print()

Query: What is an intelligent agent?
Retrieved 6 chunks:

--- Result 1 (source: Lecture 1 - Introduction to Artificial Intelligence.pdf, page: 12) ---
An intelligent agent is one that acts rationally with respect to its goals. • An agent is a function F from percept environments to actions, i.e., F: 𝑃∗ → 𝐴
─ Rational behavior: doing the right thing
─ The right thing: that which is expected to maximize goal achievement, given the 
available informa

--- Result 2 (source: Lecture 2 - Intelligent Agents.pdf, page: 2) ---
Worcester Polytechnic Institute
What is an Intelligent Agent? Input
Stimulus
Percept
Output
Response
Action
(through sensors) (through actuators)
environment
• The Intelligent Agent NOT ONLY interacts with the environment BUT ALSO interacts with other Intelligent Agents, and 
then reacts to the chan

--- Result 3 (source: Lecture 2 - Intelligent Agents.pdf, page: 6) ---
Worcester Polytechnic Institute
What is an Intelligent Agent?

--- Result 4 (source: Lecture 1 - Introd

## RAG Chain - Retrieval + Generation

Wire the `retriever` up to a local LLM via Ollama (`llama3.2`, already pulled locally - no API key needed). The chain: question → retrieve top-k chunks → stuff them into a prompt → LLM answers grounded in that context.

In [16]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

llm = ChatOllama(model="llama3.2", temperature=0)

prompt = ChatPromptTemplate.from_template(
    """Using only the context below, write a clear, complete answer of 2-4 sentences explaining the concept in your own words. If the context doesn't contain the answer, say you don't know.

Capitalization and phrasing may differ between the question and the context (e.g. "Intelligent Agent" vs "intelligent agent") - treat these as the same concept and do not treat a case difference as the concept being undefined.

Context:
{context}

Question: {question}

Answer:"""
)


def format_docs(docs):
    seen = set()
    deduped = []
    for doc in docs:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            deduped.append(doc)
    return "\n\n".join(
        f"[{doc.metadata.get('source_file')}, page {doc.metadata.get('page')}]\n{doc.page_content}"
        for doc in deduped
    )


def normalize_question(question: str) -> str:
    """Normalize surface casing so the LLM doesn't treat e.g. 'Intelligent Agent'
    and 'intelligent agent' as different, undefined terms."""
    question = question.strip()
    return question[:1].upper() + question[1:].lower() if question else question


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough() | RunnableLambda(normalize_question)}
    | prompt
    | llm
    | StrOutputParser()
)

In [20]:
question = "What is 8-queens problem?"
answer = rag_chain.invoke(question)

print(f"Question: {question}\n")
print(f"Answer: {answer}")

Question: What is 8-queens problem?

Answer: The 8-Queens problem is a classic problem in computer science and chess, where the goal is to place eight queens on a standard 8x8 chessboard such that no two queens attack each other. The problem is solved by finding the arrangement of queens that minimizes the number of conflicts, with the optimal solution having zero conflicts.
